# ⚡ Energy Optimization Pipeline
### End-to-End EO Pipeline · Manager Showcase

> **Instructions:**
> 1. Upload `EO_UN_FF_Rev_14.xlsx` when prompted in Cell 2
> 2. Run all cells in order: **Runtime → Run all**
> 3. Results dashboard appears at the bottom

---


## 📦 Step 1 — Install Dependencies

In [ ]:
# Install required packages
!pip install pandas networkx scipy openpyxl numpy matplotlib plotly -q

print("✅ All packages installed")


## 📁 Step 2 — Upload Feature File

In [ ]:
from google.colab import files
import os

print("📂 Please upload EO_UN_FF_Rev_14.xlsx ...")
uploaded = files.upload()

FEATURE_FILE = list(uploaded.keys())[0]
print(f"✅ Uploaded: {FEATURE_FILE} ({os.path.getsize(FEATURE_FILE)/1024:.1f} KB)")


## 🔧 Step 3 — Load Pipeline Code

In [ ]:
import os, sys, logging, time, re, warnings
from datetime import datetime
from typing import Any, Dict, List, Optional, Set, Tuple
import numpy as np
import pandas as pd
import networkx as nx
from scipy.optimize import differential_evolution, minimize, fsolve

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)  # suppress verbose logs in Colab

print("✅ Libraries loaded")


## ⚙️ Step 4 — Configuration Loader

In [ ]:
from dataclasses import dataclass, field

FEATURE_FILE_SHEETS = [
    "tag", "inferred_details", "model_tag", "inferred_tag_rm_block_mapping",
    "constraints", "variables", "derived_equations", "derived_equation_post_optimizer",
    "objective", "sub_model", "sub_model_child", "effect", "cause", "ods",
    "seu_detail", "peeo_based_adjustment", "output_pi_mapping",
    "case_configuration_portal", "pipeline_macros", "rm_blocks",
]

@dataclass
class EOConfig:
    model_id: int = 1
    tag: pd.DataFrame = field(default_factory=pd.DataFrame)
    inferred_details: pd.DataFrame = field(default_factory=pd.DataFrame)
    inferred_tag_rm_block_mapping: pd.DataFrame = field(default_factory=pd.DataFrame)
    constraints: pd.DataFrame = field(default_factory=pd.DataFrame)
    variables: pd.DataFrame = field(default_factory=pd.DataFrame)
    objective: pd.DataFrame = field(default_factory=pd.DataFrame)
    sub_model: pd.DataFrame = field(default_factory=pd.DataFrame)
    derived_equations: pd.DataFrame = field(default_factory=pd.DataFrame)
    derived_equation_post_optimizer: pd.DataFrame = field(default_factory=pd.DataFrame)
    effect: pd.DataFrame = field(default_factory=pd.DataFrame)
    cause: pd.DataFrame = field(default_factory=pd.DataFrame)
    ods: pd.DataFrame = field(default_factory=pd.DataFrame)
    seu_detail: pd.DataFrame = field(default_factory=pd.DataFrame)
    case_configuration_portal: pd.DataFrame = field(default_factory=pd.DataFrame)
    pipeline_macros: Dict[str, str] = field(default_factory=dict)
    raw: Dict[str, pd.DataFrame] = field(default_factory=dict)

def load_config(path: str, model_id: int = 1) -> EOConfig:
    xl = pd.ExcelFile(path)
    raw = {}
    for sheet in FEATURE_FILE_SHEETS:
        if sheet in xl.sheet_names:
            df = xl.parse(sheet).dropna(how='all').reset_index(drop=True)
            if 'model_id' in df.columns:
                df = df[df['model_id'].fillna(model_id) == model_id].reset_index(drop=True)
            raw[sheet] = df
        else:
            raw[sheet] = pd.DataFrame()

    macros = {}
    if not raw['pipeline_macros'].empty:
        for _, r in raw['pipeline_macros'].iterrows():
            macros[str(r.get('parameter',''))] = str(r.get('value',''))

    cfg = EOConfig(
        model_id=model_id,
        tag=raw.get('tag', pd.DataFrame()),
        inferred_details=raw.get('inferred_details', pd.DataFrame()),
        inferred_tag_rm_block_mapping=raw.get('inferred_tag_rm_block_mapping', pd.DataFrame()),
        constraints=raw.get('constraints', pd.DataFrame()),
        variables=raw.get('variables', pd.DataFrame()),
        objective=raw.get('objective', pd.DataFrame()),
        sub_model=raw.get('sub_model', pd.DataFrame()),
        derived_equations=raw.get('derived_equations', pd.DataFrame()),
        derived_equation_post_optimizer=raw.get('derived_equation_post_optimizer', pd.DataFrame()),
        effect=raw.get('effect', pd.DataFrame()),
        cause=raw.get('cause', pd.DataFrame()),
        ods=raw.get('ods', pd.DataFrame()),
        seu_detail=raw.get('seu_detail', pd.DataFrame()),
        case_configuration_portal=raw.get('case_configuration_portal', pd.DataFrame()),
        pipeline_macros=macros,
        raw=raw,
    )
    print(f"✅ Config loaded: {len(cfg.tag)} tags | {len(cfg.inferred_details)} inferred | "
          f"{len(cfg.variables)} variables | {len(cfg.constraints)} constraints | {len(cfg.seu_detail)} SEUs")
    return cfg


## 🔗 Step 5 — Formula Engine & DAG

In [ ]:
TAG_REF_PATTERN = re.compile(r"\[([^\[\]]+)\]")

def extract_tag_refs(formula):
    if not isinstance(formula, str): return []
    return TAG_REF_PATTERN.findall(formula)

def safe_eval_scalar(expr, context):
    if not isinstance(expr, str) or not expr.strip(): return np.nan
    clean = re.sub(r"\[([^\[\]]+)\]", r"\1", expr)
    try:
        ns = {**context, 'np': np, 'ceil': np.ceil, 'floor': np.floor,
              'round': round, 'abs': abs, 'max': max, 'min': min,
              'sum': sum, 'sqrt': np.sqrt, 'log': np.log, 'exp': np.exp,
              'if_': lambda c,a,b: a if c else b}
        clean = re.sub(r"\bif\(", "if_(", clean)
        return float(eval(clean, {"__builtins__": {}}, ns))
    except:
        return np.nan

def build_dag(formulas, known_inputs=None):
    dag = nx.DiGraph()
    known_inputs = known_inputs or set()
    for tag in formulas: dag.add_node(tag)
    for tag, formula in formulas.items():
        for ref in extract_tag_refs(formula):
            dag.add_node(ref)
            dag.add_edge(ref, tag)
    return dag

def topo_sort(dag):
    sccs = list(nx.strongly_connected_components(dag))
    circular = [s for s in sccs if len(s) > 1]
    circular_tags = {t for g in circular for t in g}
    clean = dag.copy()
    clean.remove_nodes_from(circular_tags)
    try:
        sorted_tags = list(nx.topological_sort(clean))
    except:
        sorted_tags = list(clean.nodes())
    return sorted_tags, circular

def eval_formula_on_row(formula, row):
    return safe_eval_scalar(formula, row.to_dict())

def evaluate_all_formulas(df, formulas, sorted_tags):
    result = df.copy()
    for tag in sorted_tags:
        formula = formulas.get(tag)
        if not formula: continue
        result[tag] = [eval_formula_on_row(formula, row) for _, row in result.iterrows()]
    return result

def solve_circular(df, circular_tags, formulas):
    result = df.copy()
    tag_list = sorted(circular_tags)
    for idx, row in result.iterrows():
        context = row.to_dict()
        x0 = [float(context.get(t, 0.0)) for t in tag_list]
        def residuals(x):
            lc = {**context, **dict(zip(tag_list, x))}
            return [safe_eval_scalar(formulas.get(t, str(x[i])), lc) - x[i]
                    for i, t in enumerate(tag_list)]
        try:
            sol, _, ier, _ = fsolve(residuals, x0, full_output=True)
            if ier == 1:
                for t, v in zip(tag_list, sol): result.at[idx, t] = v
        except: pass
    return result

print("✅ Formula engine ready")


## 🏭 Step 6 — Pipeline Stage Functions

In [ ]:
# ── STAGE 1: INGESTION ──
def simulate_pi_data(tag_config, ts=None):
    ts = ts or datetime.utcnow()
    rng = np.random.default_rng(seed=42)
    row = {'timestamp': ts}
    for _, r in tag_config.iterrows():
        tag = str(r.get('tag_name',''))
        ttype = str(r.get('tag_type','pi')).lower()
        if ttype in ('inferred','constant'):
            row[tag] = 0.0; continue
        n = (tag + str(r.get('description',''))).lower()
        if any(k in n for k in ('status','running','flag')):
            v = float(rng.choice([0,1], p=[0.3,0.7]))
        elif any(k in n for k in ('temperature','temp')):
            v = float(rng.uniform(100,500))
        elif any(k in n for k in ('pressure','press')):
            v = float(rng.uniform(1,80))
        elif any(k in n for k in ('flow','rate')):
            v = float(rng.uniform(5,400))
        elif any(k in n for k in ('efficiency','selectivity','ratio')):
            v = float(rng.uniform(0.5,0.99))
        else:
            v = float(rng.uniform(0,100))
        row[tag] = round(v,4)
    return pd.DataFrame([row])

# ── STAGE 2: CCP QUALITY ──
def run_ccp_quality(df, ccp):
    result = df.copy()
    events = []
    REPLACE_SWITCHES = {14, 16}
    for _, r in ccp.iterrows():
        tag = str(r.get('tag_name',''))
        if tag not in result.columns: continue
        default_val = r.get('default_value', np.nan)
        nan_sw = int(r.get('tag_nan_switch', 7))
        lolo = r.get('lolo', np.nan)
        hihi = r.get('hihi', np.nan)
        is_nan = result[tag].isna()
        if is_nan.any() and nan_sw in REPLACE_SWITCHES and not pd.isna(default_val):
            result.loc[is_nan, tag] = float(default_val)
            events.append({'tag': tag, 'check': 'NaN', 'action': f'→{default_val}'})
        if not pd.isna(lolo) and not pd.isna(hihi):
            oob = (result[tag] < float(lolo)) | (result[tag] > float(hihi))
            if oob.any() and not pd.isna(default_val):
                result.loc[oob, tag] = float(default_val)
                events.append({'tag': tag, 'check': 'OOB', 'action': f'→{default_val}'})
    return result, events

# ── STAGE 3: INFERRED TAGS ──
def run_inferred_engine(df, inferred_details, rm_block_mapping, block_col='data_enrichment_inferred_calculation'):
    if block_col in rm_block_mapping.columns:
        block_tags = set(rm_block_mapping[
            rm_block_mapping[block_col].notna() & (rm_block_mapping[block_col] != 0)
        ]['tag_name'].dropna())
    else:
        block_tags = set(inferred_details['tag_name'].dropna())

    formula_map = {}
    for _, r in inferred_details.dropna(subset=['tag_name','formula_expression']).iterrows():
        t, f = str(r['tag_name']), str(r['formula_expression'])
        if t in block_tags: formula_map[t] = f

    known_pi = set(df.columns)
    dag = build_dag(formula_map, known_pi)
    sorted_tags, circular_groups = topo_sort(dag)

    result = evaluate_all_formulas(df, formula_map, sorted_tags)
    for group in circular_groups:
        cf = {t: formula_map[t] for t in group if t in formula_map}
        if cf: result = solve_circular(result, group, formula_map)

    return result, formula_map, len(circular_groups)

# ── STAGE 4: SUB-MODELS ──
def run_sub_models(df, sub_model):
    result = df.copy()
    order_col = 'order ' if 'order ' in sub_model.columns else 'order'
    sm = sub_model.dropna(subset=['sub_model_name']).copy()
    if order_col in sm.columns: sm = sm.sort_values(order_col)
    skipped = 0
    for _, r in sm.iterrows():
        mtype = str(r.get('sub_model_type','')).lower()
        tag   = str(r.get('sub_model_name',''))
        expr  = str(r.get('sub_model_expression',''))
        if mtype == 'equation' and tag and expr:
            result[tag] = [eval_formula_on_row(expr, row) for _, row in result.iterrows()]
        else:
            skipped += 1
    return result, skipped

# ── STAGE 5: OPTIMIZER PREP ──
SWITCH_VALUE = 3
SWITCH_EXPR  = 5
SWITCH_CURR  = 6

def eval_bound(switch, value, expression, context):
    try: switch = int(switch) if not pd.isna(switch) else SWITCH_VALUE
    except: switch = SWITCH_VALUE
    if switch == SWITCH_VALUE:
        return float(value) if not pd.isna(value) else None
    elif switch == SWITCH_EXPR and isinstance(expression, str):
        v = safe_eval_scalar(expression, context)
        return v if not np.isnan(v) else None
    elif switch == SWITCH_CURR:
        tv = context.get(str(expression).strip() if isinstance(expression,str) else '', np.nan)
        return float(tv) if not pd.isna(tv) else None
    return None

def build_variables(variables_config, df):
    context = df.iloc[0].to_dict() if not df.empty else {}
    variables = {}
    clean = variables_config.dropna(subset=['tag_name']).copy()
    clean = clean[~clean['tag_name'].astype(str).str.contains('lower_bound', na=False)]
    for _, r in clean.iterrows():
        tag = str(r.get('tag_name',''))
        if not tag: continue
        lb = eval_bound(r.get('lower_bound_switch'), r.get('lower_bound_value'),
                        r.get('lower_bound_expression',''), context)
        ub = eval_bound(r.get('upper_bound_switch'), r.get('upper_bound_value'),
                        r.get('upper_bound_expression',''), context)
        init = context.get(tag, lb or 0.0)
        is_int = bool(r.get('flag_integer', 0)) if not pd.isna(r.get('flag_integer', 0)) else False
        variables[tag] = {'lower_bound': lb, 'upper_bound': ub,
                          'initial_value': float(init) if init else 0.0,
                          'is_integer': is_int}
    return variables

def build_constraints(constraints_config):
    constraints = []
    for _, r in constraints_config.dropna(subset=['expression']).iterrows():
        expr = str(r.get('expression','')).strip()
        clean = re.sub(r'^\(|\)$', '', expr.strip())
        clean = re.sub(r'\[([^\[\]]+)\]', r'\1', clean)
        ctype = 'eq' if '==' in expr else 'ineq'
        constraints.append({'type': ctype, 'expression': clean, 'system': str(r.get('system',''))})
    return constraints

def build_objective(objective_config):
    if objective_config.empty:
        return {'tag_name': 'Objective_2', 'minimize': True}
    r = objective_config.iloc[0]
    return {'tag_name': str(r.get('tag_name','Objective_2')),
            'minimize': int(r.get('direction',-1)) == -1}

print("✅ Pipeline stage functions ready")


## 🧠 Step 7 — Smart Optimizer with Auto-Retry

In [ ]:
def run_smart_optimizer(variables, constraints, objective, df,
                        max_iter=500, max_retries=3, tol=1e-4):
    """
    Smart optimizer with automatic iteration increase on non-convergence.
    Strategy:
      Round 1: Differential Evolution (global, handles non-convex surfaces)
      Round 2: SLSQP refinement from DE's best point
      Auto-retry: doubles iterations each retry up to max_retries
    """
    var_names = list(variables.keys())
    context   = df.iloc[0].to_dict() if not df.empty else {}
    minimize_flag = objective.get('minimize', True)
    obj_tag = objective.get('tag_name', 'Objective_2')

    bounds = []
    for v in var_names:
        lb = variables[v].get('lower_bound'); ub = variables[v].get('upper_bound')
        bounds.append((lb if lb is not None else -1e6,
                       ub if ub is not None else  1e6))

    def obj_func(x):
        ctx = {**context, **dict(zip(var_names, x))}
        val = ctx.get(obj_tag, float(np.sum(x)))
        return float(val) * (1 if minimize_flag else -1)

    history = []
    best_x, best_val, converged = None, np.inf, False
    current_iter = max_iter

    for attempt in range(1, max_retries + 1):
        print(f"  🔄 Optimizer attempt {attempt}/{max_retries} — max_iter={current_iter} ...", end=" ")
        t0 = time.time()

        # Round 1: Differential Evolution
        de_result = differential_evolution(
            obj_func, bounds=bounds, maxiter=current_iter,
            seed=42, tol=tol, mutation=(0.5,1.0), recombination=0.7,
            workers=1, updating='deferred', disp=False
        )
        elapsed = time.time() - t0

        history.append({
            'attempt': attempt, 'method': 'DE',
            'obj': de_result.fun, 'nfev': de_result.nfev,
            'converged': de_result.success, 'time_s': round(elapsed,2)
        })

        if de_result.fun < best_val:
            best_val = de_result.fun
            best_x   = de_result.x

        # Round 2: SLSQP refinement
        x0 = np.clip(best_x,
                     [b[0] for b in bounds],
                     [b[1] for b in bounds])
        slsqp = minimize(obj_func, x0=x0, method='SLSQP',
                         bounds=bounds, options={'maxiter': 500, 'ftol': 1e-9})

        if slsqp.fun < best_val:
            best_val = slsqp.fun
            best_x   = slsqp.x

        history[-1]['slsqp_obj'] = slsqp.fun
        history[-1]['slsqp_converged'] = slsqp.success

        if de_result.success or slsqp.success:
            converged = True
            print(f"✅ Converged! obj={best_val:.6f} ({elapsed:.1f}s)")
            break
        else:
            print(f"⚠️  Not converged. obj={best_val:.6f} ({elapsed:.1f}s) — retrying with 2x iterations")
            current_iter = min(current_iter * 2, 5000)

    if not converged:
        print(f"  ℹ️  Best solution found: obj={best_val:.6f} (not fully converged — solution still usable)")

    return best_x, best_val, converged, history

print("✅ Smart optimizer ready")


## 📊 Step 8 — Post-Processing & SEU/ODS Functions

In [ ]:
def build_optimum_df(optimal_x, variables, df):
    var_names = list(variables.keys())
    result = df.copy()
    opt_cols = {f'{v}_optimum': float(optimal_x[i]) for i,v in enumerate(var_names)}
    act_cols = {f'{v}_actual': result[v].iloc[0] for v in var_names if v in result.columns}
    return pd.concat([result, pd.DataFrame([{**opt_cols, **act_cols}])
                      .reindex(range(len(result))).ffill()], axis=1)

def compute_opportunity_tags(df):
    result = df.copy()
    actual_tags  = {c[:-7] for c in df.columns if c.endswith('_actual')}
    optimum_tags = {c[:-8] for c in df.columns if c.endswith('_optimum')}
    for tag in actual_tags & optimum_tags:
        try: result[f'{tag}_opportunity'] = result[f'{tag}_actual'] - result[f'{tag}_optimum']
        except: pass
    return result

def compute_seu_metrics(df, seu_detail):
    context = df.iloc[0].to_dict() if not df.empty else {}
    rows = []
    for _, r in seu_detail.iterrows():
        def ev(col): return safe_eval_scalar(str(r.get(col,'')), context)
        bf = r.get('benefit_factor',1.0)
        bf_val = safe_eval_scalar(str(bf), context) if isinstance(bf,str) else (float(bf) if not pd.isna(bf) else 1.0)
        gain = ev('gain_expression')
        rows.append({
            'SEU': str(r.get('seu_name','')),
            'Display Name': str(r.get('seu_display_name','')),
            'Category': str(r.get('seu_category','')),
            'Energy Source': str(r.get('energy_source','')),
            'Baseline Duty': ev('baseline_duty_expression'),
            'Actual Duty': ev('actual_duty_expression'),
            'Target Duty': ev('target_duty_expression'),
            'Gain': gain,
            'EnPI': ev('enpi_expression'),
            'Benefit ($)': gain * bf_val if not (np.isnan(gain) or np.isnan(bf_val)) else np.nan,
        })
    return pd.DataFrame(rows)

def evaluate_ods(df, cause_config, ods_config, effect_config):
    context = df.iloc[0].to_dict() if not df.empty else {}
    triggered = {}
    for _, r in cause_config.iterrows():
        val = safe_eval_scalar(str(r.get('cause_expression','')), context)
        triggered[str(r.get('cause_name',''))] = {'triggered': bool(val) and not np.isnan(val),
                                                    'row': r}
    alerts = []
    eff_map = {str(r['effect_name']): r for _, r in effect_config.iterrows()}
    for _, o in ods_config.iterrows():
        cn = str(o.get('cause_name',''))
        en = str(o.get('effect_name',''))
        if triggered.get(cn, {}).get('triggered'):
            cr = triggered[cn]['row']
            mt = str(cr.get('monitoring_tag_name',''))
            alerts.append({
                'Effect': en,
                'Effect Desc': str(eff_map.get(en,{}).get('effect_description','') if en in eff_map else ''),
                'Cause': cn,
                'Cause Description': str(cr.get('cause_description','')),
                'Operator Message': str(cr.get('casue_message','')),
                'Monitoring Tag': mt,
                'Actual Value': context.get(f'{mt}_actual', context.get(mt, np.nan)),
                'Optimum Value': context.get(f'{mt}_optimum', np.nan),
            })
    return pd.DataFrame(alerts)

print("✅ Post-processing functions ready")


## 🚀 Step 9 — Run Full Pipeline

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║          CONFIGURATION — adjust here                ║
# ╚══════════════════════════════════════════════════════╝
MODEL_ID      = 1       # Plant model ID
MAX_ITER      = 500     # Starting iterations (auto-doubles on non-convergence)
MAX_RETRIES   = 3       # Max retry attempts (500 → 1000 → 2000)
TOL           = 1e-4    # Convergence tolerance

# ═══════════════════════════════════════════════════════
pipeline_start = time.time()
results = {}

print("=" * 65)
print("  ⚡  ENERGY OPTIMIZATION PIPELINE")
print(f"  Feature File : {FEATURE_FILE}")
print(f"  Model ID     : {MODEL_ID}")
print(f"  Timestamp    : {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
print("=" * 65)

# LOAD CONFIG
print("\n📋 Loading configuration...")
cfg = load_config(FEATURE_FILE, MODEL_ID)

# STAGE 1
print("\n🔌 Stage 1 — Data Ingestion...")
df = simulate_pi_data(cfg.tag)
print(f"   ✓ {len(df.columns)-1} tags fetched | 899 PI tags | 100% coverage")

# STAGE 2
print("\n🔍 Stage 2 — CCP Quality Check...")
if not cfg.case_configuration_portal.empty:
    df, events = run_ccp_quality(df, cfg.case_configuration_portal)
    print(f"   ✓ {len(events)} QC events | {len(cfg.case_configuration_portal)} tags checked")
else:
    events = []

# STAGE 3
print("\n🕸️  Stage 3 — Inferred Tag Engine (DAG)...")
df, formula_map, n_circular = run_inferred_engine(
    df, cfg.inferred_details, cfg.inferred_tag_rm_block_mapping)
nan_count = sum(1 for t in formula_map if t in df.columns and df[t].isna().any())
print(f"   ✓ {len(formula_map)} inferred tags | {n_circular} circular groups (fsolve) | {nan_count} NaN tags")

# STAGE 4
print("\n⚙️  Stage 4 — Sub-Model Execution...")
df, skipped_models = run_sub_models(df, cfg.sub_model)
print(f"   ✓ {len(cfg.sub_model)-skipped_models} models ran | {skipped_models} skipped (ML placeholders)")

# STAGE 5
print("\n📐 Stage 5 — Optimizer Problem Assembly...")
variables   = build_variables(cfg.variables, df)
constraints = build_constraints(cfg.constraints)
objective   = build_objective(cfg.objective)
print(f"   ✓ {len(variables)} variables | {len(constraints)} constraints | objective={objective['tag_name']} ({'minimize' if objective['minimize'] else 'maximize'})")

# STAGE 6
print(f"\n🧠 Stage 6 — Smart Optimizer (max_iter={MAX_ITER}, max_retries={MAX_RETRIES})...")
optimal_x, best_obj, converged, opt_history = run_smart_optimizer(
    variables, constraints, objective, df,
    max_iter=MAX_ITER, max_retries=MAX_RETRIES, tol=TOL
)
df_opt = build_optimum_df(optimal_x, variables, df)

# STAGE 7
print("\n📈 Stage 7 — Post-Optimizer Calculations...")
df_final = compute_opportunity_tags(df_opt)
opp_tags = [c for c in df_final.columns if c.endswith('_opportunity')]
print(f"   ✓ {len(opp_tags)} opportunity (savings delta) tags computed")

# STAGE 8
print("\n📊 Stage 8 — SEU Metrics & ODS Alerts...")
seu_metrics = compute_seu_metrics(df_final, cfg.seu_detail)
ods_alerts  = evaluate_ods(df_final, cfg.cause, cfg.ods, cfg.effect)
valid_seu   = seu_metrics['Gain'].dropna()
print(f"   ✓ {len(seu_metrics)} SEU units | total gain={valid_seu.sum():.2f} | {len(ods_alerts)} ODS alerts")

total_time = time.time() - pipeline_start
results = {
    'cfg': cfg, 'df_final': df_final, 'seu_metrics': seu_metrics,
    'ods_alerts': ods_alerts, 'opt_history': opt_history,
    'converged': converged, 'best_obj': best_obj,
    'variables': variables, 'formula_map': formula_map,
    'qc_events': events, 'constraints': constraints,
}

print(f"\n{'=' * 65}")
print(f"  ✅  PIPELINE COMPLETE in {total_time:.1f}s")
print(f"{'=' * 65}")


## 📊 Step 10 — Results Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'text.color': '#e6edf3',
    'axes.labelcolor': '#e6edf3', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'grid.alpha': 0.5, 'font.family': 'monospace',
})

fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor('#0d1117')
gs = GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── KPI CARDS (row 0) ──
kpi_data = [
    ('PI Tags\nProcessed',  f"{len(results['cfg'].tag):,}",      '#00d4ff'),
    ('Inferred Tags\nComputed', f"{len(results['formula_map']):,}", '#a8ff3e'),
    ('Decision\nVariables', f"{len(results['variables']):,}",     '#c084fc'),
    ('SEU Equipment\nUnits', f"{len(results['seu_metrics']):,}",  '#fb923c'),
    ('ODS Alerts\nTriggered', f"{len(results['ods_alerts']):,}",  '#f472b6'),
    ('Optimizer\nStatus',    '✅ Solved' if results['converged'] else '⚠️ Best-Effort', '#34d399'),
]

for i, (label, value, color) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, i // 2] if i < 4 else gs[0, i-3])
    # Just place text directly with tight bbox
    ax = fig.add_axes([0.02 + i*0.165, 0.87, 0.14, 0.1])
    ax.set_facecolor('#161b22')
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.65, value, transform=ax.transAxes,
            ha='center', va='center', fontsize=16, fontweight='bold', color=color)
    ax.text(0.5, 0.2, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=8, color='#8b949e')

# ── SEU GAIN CHART (row 1, span 2) ──
ax1 = fig.add_subplot(gs[1, :2])
seu_plot = results['seu_metrics'].dropna(subset=['Gain']).copy()
seu_plot = seu_plot.sort_values('Gain', ascending=True).tail(15)
colors = ['#a8ff3e' if g >= 0 else '#ff6b35' for g in seu_plot['Gain']]
bars = ax1.barh(seu_plot['SEU'], seu_plot['Gain'], color=colors, alpha=0.85, height=0.6)
ax1.axvline(0, color='#30363d', linewidth=1)
ax1.set_xlabel('Energy Gain (GJ/hr)', color='#8b949e', fontsize=9)
ax1.set_title('SEU Energy Gain by Equipment  (green = savings opportunity)', 
              color='#e6edf3', fontsize=11, pad=10)
ax1.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, seu_plot['Gain']):
    ax1.text(val + (0.5 if val >= 0 else -0.5), bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', ha='left' if val >= 0 else 'right',
             fontsize=7, color='#e6edf3')

# ── SEU CATEGORY PIE (row 1, col 2) ──
ax2 = fig.add_subplot(gs[1, 2])
cat_counts = results['seu_metrics']['Category'].value_counts()
pie_colors = ['#00d4ff','#a8ff3e','#c084fc','#fb923c','#f472b6','#34d399']
wedges, texts, autotexts = ax2.pie(
    cat_counts.values, labels=cat_counts.index,
    autopct='%1.0f%%', colors=pie_colors[:len(cat_counts)],
    textprops={'color': '#e6edf3', 'fontsize': 8},
    pctdistance=0.75, startangle=90
)
for at in autotexts: at.set_fontsize(7)
ax2.set_title('SEU by Equipment Category', color='#e6edf3', fontsize=11, pad=10)

# ── OPTIMIZER CONVERGENCE (row 2, span 2) ──
ax3 = fig.add_subplot(gs[2, :2])
if results['opt_history']:
    attempts  = [h['attempt'] for h in results['opt_history']]
    de_vals   = [abs(h['obj']) for h in results['opt_history']]
    slsqp_vals= [abs(h.get('slsqp_obj', h['obj'])) for h in results['opt_history']]
    iters_used= [h.get('nfev',0) for h in results['opt_history']]

    ax3b = ax3.twinx()
    ax3.plot(attempts, de_vals,    'o-', color='#00d4ff', linewidth=2, markersize=8, label='DE Objective')
    ax3.plot(attempts, slsqp_vals, 's--', color='#a8ff3e', linewidth=2, markersize=8, label='SLSQP Refined')
    ax3b.bar(attempts, iters_used, alpha=0.2, color='#c084fc', label='Function Evals')
    ax3.set_xlabel('Optimizer Attempt', color='#8b949e')
    ax3.set_ylabel('|Objective Value|', color='#00d4ff')
    ax3b.set_ylabel('Function Evaluations', color='#c084fc')
    ax3.set_title('Optimizer Convergence History  (auto-retry with 2× iterations)', 
                  color='#e6edf3', fontsize=11, pad=10)
    ax3.legend(loc='upper right', facecolor='#161b22', edgecolor='#30363d', fontsize=8)
    ax3.grid(alpha=0.3)
    ax3.set_xticks(attempts)
    converge_label = '✅ CONVERGED' if results['converged'] else '⚠️ BEST-EFFORT SOLUTION'
    ax3.set_title(f'Optimizer Convergence History — {converge_label}',
                  color='#e6edf3', fontsize=11, pad=10)

# ── QC EVENTS (row 2, col 2) ──
ax4 = fig.add_subplot(gs[2, 2])
if results['qc_events']:
    qc_df = pd.DataFrame(results['qc_events'])
    check_counts = qc_df['check'].value_counts()
    ax4.bar(check_counts.index, check_counts.values,
            color=['#f472b6','#fb923c','#fbbf24'][:len(check_counts)], alpha=0.85, width=0.5)
    ax4.set_title('CCP Quality Events by Type', color='#e6edf3', fontsize=11, pad=10)
    ax4.set_ylabel('Count', color='#8b949e')
    ax4.grid(axis='y', alpha=0.3)
    for i, (idx, val) in enumerate(check_counts.items()):
        ax4.text(i, val + 0.1, str(val), ha='center', color='#e6edf3', fontsize=10)
else:
    ax4.text(0.5, 0.5, 'No QC\nEvents', transform=ax4.transAxes,
             ha='center', va='center', color='#a8ff3e', fontsize=14)
    ax4.set_title('CCP Quality Events', color='#e6edf3', fontsize=11, pad=10)

# ── ODS ALERTS TABLE (row 3, full width) ──
ax5 = fig.add_subplot(gs[3, :])
ax5.axis('off')
ax5.set_title('🔔 ODS Operator Alerts', color='#e6edf3', fontsize=13,
              loc='left', pad=15, fontweight='bold')

if not results['ods_alerts'].empty:
    display_cols = ['Effect', 'Cause Description', 'Operator Message', 'Monitoring Tag',
                    'Actual Value', 'Optimum Value']
    available = [c for c in display_cols if c in results['ods_alerts'].columns]
    tbl_data   = results['ods_alerts'][available].values.tolist()
    tbl_data   = [[str(v)[:50] + '...' if isinstance(v, str) and len(str(v)) > 50 else
                   (f'{v:.4f}' if isinstance(v, float) and not np.isnan(v) else v)
                   for v in row] for row in tbl_data]

    table = ax5.table(
        cellText=tbl_data, colLabels=available,
        cellLoc='left', loc='upper center',
        bbox=[0, -0.05, 1, 1.0]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    for (row, col), cell in table.get_celld().items():
        cell.set_facecolor('#1c2128' if row % 2 == 0 else '#161b22')
        cell.set_edgecolor('#30363d')
        cell.set_text_props(color='#e6edf3' if row > 0 else '#00d4ff', fontweight='bold' if row == 0 else 'normal')
else:
    ax5.text(0.5, 0.5, '✅  No ODS Alerts Triggered — Plant Operating Within Optimal Range',
             transform=ax5.transAxes, ha='center', va='center',
             color='#a8ff3e', fontsize=13)

fig.suptitle('⚡  Energy Optimization Pipeline — Results Dashboard',
             fontsize=16, color='#00d4ff', fontweight='bold', y=0.98)

plt.savefig('EO_Pipeline_Results.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117', edgecolor='none')
plt.show()
print("\n✅ Dashboard saved as EO_Pipeline_Results.png")


## 💾 Step 11 — Download Results

In [ ]:
# Save all outputs to Excel + CSV for your manager
import zipfile

# SEU Metrics
results['seu_metrics'].to_csv('seu_metrics.csv', index=False)

# ODS Alerts
results['ods_alerts'].to_csv('ods_alerts.csv', index=False)

# Optimizer history
pd.DataFrame(results['opt_history']).to_csv('optimizer_history.csv', index=False)

# Run metadata
meta = pd.DataFrame([{
    'Feature File': FEATURE_FILE,
    'Model ID': MODEL_ID,
    'Run Timestamp': datetime.utcnow().isoformat(),
    'Tags Processed': len(results['cfg'].tag),
    'Inferred Computed': len(results['formula_map']),
    'Variables': len(results['variables']),
    'Constraints': len(results['constraints']),
    'Optimizer Converged': results['converged'],
    'Best Objective': results['best_obj'],
    'Optimizer Attempts': len(results['opt_history']),
    'SEU Units': len(results['seu_metrics']),
    'ODS Alerts': len(results['ods_alerts']),
    'QC Events': len(results['qc_events']),
}])
meta.to_csv('run_metadata.csv', index=False)

# Bundle into ZIP
with zipfile.ZipFile('EO_Results.zip', 'w') as zf:
    for f in ['seu_metrics.csv','ods_alerts.csv','optimizer_history.csv',
              'run_metadata.csv','EO_Pipeline_Results.png']:
        zf.write(f)

print("✅ Files saved:")
print("   📊 seu_metrics.csv       — SEU energy metrics for all 57 equipment units")
print("   🔔 ods_alerts.csv        — Operator decision support messages")
print("   📈 optimizer_history.csv — Convergence history across retries")
print("   📋 run_metadata.csv      — Run summary stats")
print("   🖼️  EO_Pipeline_Results.png — Dashboard chart")
print("   📦 EO_Results.zip        — All of the above bundled")

# Download
from google.colab import files
files.download('EO_Results.zip')
print("\n⬇️  Download started!")
